In [53]:
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import numpy as np
import tqdm
from thefuzz import fuzz, process
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [54]:
import sys
print(sys.path[0])


c:\Users\morit\anaconda3\envs\DataMining\python313.zip


In [55]:
url = "https://raw.githubusercontent.com/LiberoBiagi/ML_Nova_IMS_25-26/main/train.csv"

df = pd.read_csv("train.csv")
df.head()


,carID,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,69512,VW,Golf,2016.0,22290,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.000000,0.0
1,53000,Toyota,Yaris,2019.0,13790,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.000000,0.0
2,6366,Audi,Q2,2019.0,24990,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.000000,0.0
3,29021,Ford,FIESTA,2018.0,12500,anual,9102.0,Petrol,145.0,65.700000,1.0,50.0,-2.340306,0.0
4,10062,BMW,2 Series,2019.0,22995,Manual,1000.0,Petrol,145.0,42.800000,1.5,97.0,3.000000,0.0


# Data Exploration 

In [56]:
print(df.shape)

(75973, 14)


In [57]:
#df.carID.count() # No duplicates for CarID

In [58]:
df.describe()

,carID,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
count,75973.000000,74482.000000,75973.000000,74510.000000,68069.000000,68047.000000,74457.000000,74449.000000,74423.000000,74425.0
mean,37986.000000,2017.096611,16881.889553,23004.184088,120.329078,55.152666,1.660136,64.590667,1.994580,0.0
std,21931.660338,2.208704,9736.926322,22129.788366,65.521176,16.497837,0.573462,21.021065,1.472981,0.0
min,0.000000,1970.000000,450.000000,-58540.574478,-91.121630,-43.421768,-0.103493,1.638913,-2.345650,0.0
25%,18993.000000,2016.000000,10200.000000,7423.250000,125.000000,46.300000,1.200000,47.000000,1.000000,0.0
50%,37986.000000,2017.000000,14699.000000,17300.000000,145.000000,54.300000,1.600000,65.000000,2.000000,0.0
75%,56979.000000,2019.000000,20950.000000,32427.500000,145.000000,62.800000,2.000000,82.000000,3.000000,0.0
max,75972.000000,2024.121759,159999.000000,323000.000000,580.000000,470.800000,6.600000,125.594308,6.258371,0.0


In [59]:
df.dtypes

carID               int64
Brand              object
model              object
year              float64
price               int64
transmission       object
mileage           float64
fuelType           object
tax               float64
mpg               float64
engineSize        float64
paintQuality%     float64
previousOwners    float64
hasDamage         float64
dtype: object

In [60]:
df.isna().sum()

carID                0
Brand             1521
model             1517
year              1491
price                0
transmission      1522
mileage           1463
fuelType          1511
tax               7904
mpg               7926
engineSize        1516
paintQuality%     1524
previousOwners    1550
hasDamage         1548
dtype: int64

# Data Cleaning

# Data Preprocessing Overview

## Preprocessing by Column

| Column | General Preprocessing | Imputation Method | Notes |
|--------|----------------------|-------------------|-------|
| **carID** | None | No imputation (identifier) | Unique identifier, not used in modeling |
| **Brand** | - Lowercase & strip whitespace<br>- Dictionary mapping for typo correction<br>- Fill missing based on model mode | KNNImputer (k=10, distance-weighted) | Imputed using similar cars based on all features, this is the case for when model and brand are missing |
| **model** | - Lowercase & strip whitespace<br>- Vectorized fuzzy matching against reference list<br>- Separate handling for 2-letter models | Brand-specific KNNImputer (k=10) | Imputed within each brand to ensure brand-model consistency |
| **year** | - Round to integer | IterativeImputer (mean) | Imputed using relationships with all other features |
| **transmission** | - Lowercase & strip whitespace<br>- Vectorized fuzzy matching against reference list<br>- Replace "unknown" and "other" with missing | KNNImputer (k=10, distance-weighted) | Imputed using similar cars based on all features |
| **mileage** | - Round to integer | IterativeImputer (mean) | Imputed using relationships with all other features |
| **fuelType** | - Lowercase & strip whitespace<br>- Vectorized fuzzy matching against reference list<br>- Replace "other" with missing | KNNImputer (k=10, distance-weighted) | Imputed using similar cars based on all features |
| **tax** | - Round to integer | IterativeImputer (mean) | Imputed using relationships with all other features |
| **mpg** | - Round to 1 decimal place | IterativeImputer (mean) | Imputed using relationships with all other features |
| **engineSize** | - Round to 1 decimal place | IterativeImputer (mean) | Imputed using relationships with all other features |
| **paintQuality%** | - Round to integer<br>- Fix data entry errors:<br>&nbsp;&nbsp;• Values <4: multiply by 10<br>&nbsp;&nbsp;• Values >100: subtract 100 | IterativeImputer (mean) | Imputed using relationships with all other features |
| **previousOwners** | - Round to integer<br>- Take absolute value | IterativeImputer (mean) | Imputed using relationships with all other features |
| **hasDamage** | - Convert to boolean<br>- Invert to create `stated_no_damage`<br>- Drop original column | No imputation needed | Transformed to `stated_no_damage` (True/False) |


- **Data Leakage Note**: All imputers and encoders are fitted **only on training data**, then transformed on both train and test sets


Brands <br>
We decided to do "manual" brand mapping because it gives the biggest controll factor while the number of values is managable. 
Also with some of the brand names beeing very short (e.g. vw) fuzzy algorithms would perform with reduced accuracy

Damage column <br>
This is a column that is filled by the customer prior to inspection. If the car has no damage the customer fills it in, the other values are left empty. Since it is impossible to determine the level of damage from all the other datapoints we decided to make changes to the column hasDamage. Mainly we change the purpose of the column to assesing if the customer stated that his car has no damage, in this case true (previous 0), else fales (previous missing).

In [1]:
import pandas as pd
from thefuzz import fuzz, process
import numpy as np

# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer


def clean_car_data(df_train, df_test, fast=True):
    """
    Clean and impute car data for both training and test sets.
    
    IMPORTANT: All imputers and encoders are fit ONLY on training data to prevent data leakage.
    
    Parameters:
    -----------
    df_train : pd.DataFrame
        Training dataset
    df_test : pd.DataFrame
        Test dataset
    fast : bool, default=True
        If True, uses BayesianRidge for imputation (fast).
        If False, uses RandomForestRegressor (slower but potentially better accuracy)
        
    Returns:
    --------
    df_train_cleaned : pd.DataFrame
        Cleaned and imputed training data
    df_test_cleaned : pd.DataFrame
        Cleaned and imputed test data
    """
    
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
    
    def process_dataframe(df):
        """
        Apply string cleaning, brand/model corrections, and fuzzy matching.
        These operations don't require fitting on training data.
        """
        df = df.copy()
        
        # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
        df["Brand"] = df["Brand"].str.lower().str.strip()
        df["model"] = df["model"].str.lower().str.strip()
        df["transmission"] = df["transmission"].str.lower().str.strip()
        df["fuelType"] = df["fuelType"].str.lower().str.strip()
        
        # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
        df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
        
        # 1.1 Fixing brands
        df["Brand"] = df["Brand"].map(brand_mapping)
        
        # 1.2 Fixing models
        # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
        # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
        
        # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
        
        # Models - handle different lengths separately
        unique_models = df["model"].unique()
        model_lookup = {}
        for val in unique_models:
            if pd.isna(val) or val == "NaN":
                model_lookup[val] = "NaN"
            elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
                model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
            elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
                model_lookup[val] = process.extractOne(val, short_models)[0]
            else:  # We can define models with only one letter
                model_lookup[val] = "NaN"
        df["model"] = df["model"].map(model_lookup)
        
        # Transmission
        unique_trans = df["transmission"].unique()
        trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
        df["transmission"] = df["transmission"].map(trans_lookup)
        
        # FuelType
        unique_fuel = df["fuelType"].unique()
        fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
        df["fuelType"] = df["fuelType"].map(fuel_lookup)
        
        # Convert the str NaN values back to pd.NA for easier further processing and readability
        df["model"] = df["model"].replace("NaN", pd.NA)
        df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
        df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
        
        # Get the most frequent brand for each model -> returns df with model and brand
        brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
        df = pd.merge(df, brand_models, on="model", how="left", suffixes=('', '_mode'))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
        
        df["Brand"] = df["Brand"].fillna(df['Brand_mode'])  # rename new column
        df.drop('Brand_mode', axis=1, inplace=True)  # remove the old brand column
        
        # Cleaning numeric columns
        df['year'] = df['year'].round(0)
        
        # Create the new column
        df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
        df = df.drop(["hasDamage"], axis=1)
        
        return df
    
    # Apply preprocessing to both datasets
    df_train = process_dataframe(df_train)
    df_test = process_dataframe(df_test)
    
    # ============================================================================
    # SECTION 3: ENCODING (FIT ON TRAIN, TRANSFORM BOTH) -> Risk of data leakage
    # ============================================================================
    
    def encode_dataframe(df, encoders=None, fit=False):
        """
        Encode categorical columns using LabelEncoder.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Dataframe to encode
        encoders : dict or None
            Dictionary of fitted encoders. If None and fit=True, new encoders are created
        fit : bool
            If True, fit encoders on this data. If False, use provided encoders
            
        Returns:
        --------
        df_encoded : pd.DataFrame
            Encoded dataframe
        encoders : dict (only if fit=True)
            Dictionary of fitted encoders
        """
        df = df.copy()
        
        # Fill the missing values with String value -> this is important for the encoding as it only works with one datatype per column
        df["Brand"] = df["Brand"].fillna("MISSING_VALUE")
        df["model"] = df["model"].fillna("MISSING_VALUE")
        df["transmission"] = df["transmission"].fillna("MISSING_VALUE")
        df["fuelType"] = df["fuelType"].fillna("MISSING_VALUE")
        
        if fit:
            # Train and apply the encoder (ONLY ON TRAINING DATA)
            brand_labels = LabelEncoder()
            model_labels = LabelEncoder()
            transmission_labels = LabelEncoder()
            fuelType_labels = LabelEncoder()
            
            df['brand_encoded'] = brand_labels.fit_transform(df['Brand'])
            df['model_encoded'] = model_labels.fit_transform(df['model'])
            df['transmission_encoded'] = transmission_labels.fit_transform(df['transmission'])
            df['fuelType_encoded'] = fuelType_labels.fit_transform(df['fuelType'])
            
            encoders = {
                'brand': brand_labels,
                'model': model_labels,
                'transmission': transmission_labels,
                'fuelType': fuelType_labels
            }
        else:
            # Use pre-fitted encoders (FOR TEST DATA)
            df['brand_encoded'] = encoders['brand'].transform(df['Brand'])
            df['model_encoded'] = encoders['model'].transform(df['model'])
            df['transmission_encoded'] = encoders['transmission'].transform(df['transmission'])
            df['fuelType_encoded'] = encoders['fuelType'].transform(df['fuelType'])
        
        # Get the encoding for "MISSING_VALUE"
        # And replace the encoding for missing values with NaN
        
        # Brand column
        missing_code = encoders['brand'].transform(["MISSING_VALUE"])[0]
        df['brand_encoded'] = df['brand_encoded'].replace(missing_code, np.nan)
        
        # model column
        missing_code = encoders['model'].transform(["MISSING_VALUE"])[0]
        df['model_encoded'] = df['model_encoded'].replace(missing_code, np.nan)
        
        # transmission column
        missing_code = encoders['transmission'].transform(["MISSING_VALUE"])[0]
        df['transmission_encoded'] = df['transmission_encoded'].replace(missing_code, np.nan)
        
        # fuelType column
        missing_code = encoders['fuelType'].transform(["MISSING_VALUE"])[0]
        df["fuelType_encoded"] = df["fuelType_encoded"].replace(missing_code, np.nan)
        
        if fit:
            return df, encoders
        else:
            return df
    
    # Fit encoders on training data, then apply to both datasets
    df_train_encoded, encoders = encode_dataframe(df_train, fit=True)
    df_test_encoded = encode_dataframe(df_test, encoders=encoders, fit=False)
    
    # ============================================================================
    # SECTION 4: IMPUTATION (FIT ON TRAIN, TRANSFORM BOTH) -> Risk of data leakage
    # ============================================================================
    
    # Feature definitions
    numerical_features = ["year", "mileage", "tax", "mpg", "engineSize", "paintQuality%", "previousOwners"]
    categorical_features = ["brand_encoded", "transmission_encoded", "fuelType_encoded", "model_encoded"]
    all_features = categorical_features + numerical_features
    features_without_model = ["brand_encoded", "transmission_encoded", "fuelType_encoded"] + numerical_features
    categorical_features_no_model = ["brand_encoded", "transmission_encoded", "fuelType_encoded"]
    
    def impute_dataframe(df, encoders, imputers=None, fit=False):
        """
        Impute missing values using IterativeImputer for numerical and KNNImputer for categorical.
        Also handles brand-specific model imputation.
        
        Parameters:
        -----------
        df : pd.DataFrame
            Encoded dataframe to impute
        encoders : dict
            Dictionary of label encoders for decoding
        imputers : dict or None
            Dictionary of fitted imputers. If None and fit=True, new imputers are created
        fit : bool
            If True, fit imputers on this data. If False, use provided imputers
            
        Returns:
        --------
        df_imputed : pd.DataFrame
            Imputed and decoded dataframe
        imputers : dict (only if fit=True)
            Dictionary of fitted imputers
        """
        df = df.copy()
        
        if fit:
            # Give option to perform imputation with basic estimator (BaysianRidge) or Random Forest Regressor
            # If we use the Random Forest Regressor compuationtime will increase to up to 2 min compared to 0.8s without it
            if fast:
                estimator_selection = None
            else:
                # Could use a random forrest to better deal with complex connection of the columns
                estimator_selection = RandomForestRegressor(n_estimators=20, max_depth=10, random_state=12)
            
            # Step 3: Apply IterativeImputer
            numerical_imputer = IterativeImputer(
                estimator=estimator_selection,
                max_iter=10,  # Number of imputation rounds
                random_state=12,  # For reproducibility
                initial_strategy="mean"  # Initial fill strategy
            )
            
            categorical_imputer = KNNImputer(
                n_neighbors=10,  # Consider 10 most similar cars
                weights='distance',  # Closer neighbors have more influence
                metric='nan_euclidean'  # Handles missing values properly
            )
            
            # FIT imputers on training data
            numerical_imputer.fit(df[all_features])
            categorical_imputer.fit(df[features_without_model])
            
            imputers = {
                'numerical': numerical_imputer,
                'categorical': categorical_imputer,
                'brand_imputers': {}  # Will be populated below
            }
        
        # Use all dataframe features so relationships are captured for numerical imputation
        temp_imputed = imputers['numerical'].transform(df[all_features])
        df[numerical_features] = temp_imputed[:, len(categorical_features):]
        
        # Impute categorical columns using most_freqent
        temp_cat = imputers['categorical'].transform(df[features_without_model])
        df[categorical_features_no_model] = temp_cat[:, :3].round()
        
        # Brand
        n_brand_classes = len(encoders['brand'].classes_)
        brand_clipped = np.clip(df['brand_encoded'].round(), 0, n_brand_classes - 1).astype(int)
        df['Brand'] = encoders['brand'].inverse_transform(brand_clipped)
        
        # Transmission
        n_transmission_classes = len(encoders['transmission'].classes_)
        transmission_clipped = np.clip(df['transmission_encoded'].round(), 0, n_transmission_classes - 1).astype(int)
        df['transmission'] = encoders['transmission'].inverse_transform(transmission_clipped)
        
        # FuelType
        n_fuelType_classes = len(encoders['fuelType'].classes_)
        fuelType_clipped = np.clip(df['fuelType_encoded'].round(), 0, n_fuelType_classes - 1).astype(int)
        df['fuelType'] = encoders['fuelType'].inverse_transform(fuelType_clipped)
        
        # STEP 2: Impute missing models, split by BRAND
        brands = df['Brand'].unique()
        
        for brand in brands:
            # Skip if brand is NaN
            if pd.isna(brand):
                continue
            
            # Create a sub dataframe with only one brand
            brand_mask = df['Brand'] == brand
            brand_subset = df[brand_mask].copy()
            
            # Check if this brand has any missing models
            if not brand_subset['model_encoded'].isna().any():
                continue
            
            # Find all the valid model codes for this brand
            valid_model_codes = brand_subset.loc[
                brand_subset['model_encoded'].notna(),
                'model_encoded'
            ].unique()
            
            # Impute model within this brand
            model_features = ["model_encoded"] + numerical_features + ["transmission_encoded", "fuelType_encoded", "brand_encoded"]
            
            if fit:
                # FIT a new imputer for this specific brand (ONLY ON TRAINING DATA)
                knn_model_imputer = KNNImputer(n_neighbors=10, weights='distance')
                knn_model_imputer.fit(brand_subset[model_features])
                imputers['brand_imputers'][brand] = knn_model_imputer
            else:
                # USE pre-fitted imputer for this brand (FOR TEST DATA)
                # If this brand wasn't in training data, skip imputation for this brand
                if brand not in imputers['brand_imputers']:
                    continue
                knn_model_imputer = imputers['brand_imputers'][brand]
            
            imputed = knn_model_imputer.transform(brand_subset[model_features])
            imputed_models = imputed[:, 0].round()
            
            # Validate imputed values
            for i, original_val in enumerate(brand_subset['model_encoded'].values):
                if pd.isna(original_val):
                    predicted_code = imputed_models[i]
                    
                    if predicted_code not in valid_model_codes:
                        # Use most common model for this brand
                        most_common = brand_subset['model_encoded'].mode()[0]
                        imputed_models[i] = most_common
            
            df.loc[brand_mask, "model_encoded"] = imputed_models
        
        # Decode models
        n_model_classes = len(encoders['model'].classes_)
        model_clipped = np.clip(df['model_encoded'].round(), 0, n_model_classes - 1).astype(int)
        df['model'] = encoders['model'].inverse_transform(model_clipped)
        
        # Removing encoded columns
        df.drop(["brand_encoded", "model_encoded", "transmission_encoded", "fuelType_encoded"], axis=1, inplace=True)
        
        # Final numeric cleanup and rounding
        df.year = df.year.round()
        df.mileage = abs(df.mileage.round())
        df.tax = abs(df.tax.round())
        df.mpg = df.mpg.round(1)
        df.engineSize = df.engineSize.round(1)
        
        # Assumption:
        # values <4 where entered with a wrong comma
        # values >100 are have a unnecessary leading 1
        def fix_paint_quality(x):
            if x < 4:
                return x * 10
            elif x > 100:
                return x - 100
            else:
                return x
        
        df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
        
        df.previousOwners = abs(df.previousOwners.round())
        
        if fit:
            return df, imputers
        else:
            return df
    
    # Fit imputers on training data, then apply to both datasets
    df_train_final, imputers = impute_dataframe(df_train_encoded, encoders, fit=True)
    df_test_final = impute_dataframe(df_test_encoded, encoders, imputers=imputers, fit=False)
    
    return df_train_final, df_test_final

# Preprocessing, train and selection

Drop 1-hot with the minority class

~~check the prperocessing if it's ok with the data leakage. Sava data from xtrain, apply to xval~~

Try cross val. Try basic radom search and gridsearch

dont remove 1970, use standard/robust scaler instead of minmax

MLP design, basic should have input layer = feature input, 1 hidden layer, single perceptron output layer

Apply other metrics shown in class

Models:

1. Linear Regression -> 4866.478
3. Ridge -> 4865.080
4. Lasso -> 4863.857
2. ElasticNet -> 8280.295
5. Adaline -> ask professor if sgd algos can be used
6. SVM -> 8913.638
7. Random Forest -> 3218.916
8. Single Tree -> 4390.532
9. KNN Regressor -> 4589.610
10. SGD
10. MLP


1-5 Linear

In [94]:
df["year"] = df["year"].astype(str).str[-4:].astype(int)
df["stated_no_damage"] = df["stated_no_damage"].astype(int)
df.set_index("carID", inplace=True)

ValueError: invalid literal for int() with base 10: 'NaT'

In [ ]:
df_num = df[["year", "price", "mileage", "tax", "mpg", "engineSize","paintQuality%", "previousOwners", "stated_no_damage"]]
df_num

,year,price,mileage,tax,mpg,engineSize,paintQuality%,previousOwners,stated_no_damage
carID,,,,,,,,,
69512,2016,22290,28421.0,95.95,11.4,2.0,63.0,4.0,1
53000,2019,13790,4589.0,145.00,47.9,1.5,50.0,1.0,1
6366,2019,24990,3624.0,145.00,40.9,1.5,56.0,4.0,1
29021,2018,12500,9102.0,145.00,65.7,1.0,50.0,2.0,1
10062,2019,22995,1000.0,145.00,42.8,1.5,97.0,3.0,1
...,...,...,...,...,...,...,...,...,...
37194,2015,13498,14480.0,125.00,53.3,2.0,78.0,0.0,1
6265,2013,12495,52134.0,200.00,47.9,2.0,38.0,2.0,1
54886,2017,8399,11304.0,145.00,67.0,1.0,57.0,3.0,1


In [ ]:
df_cat = df[["Brand", "model", "transmission", "fuelType"]]
df_cat

,Brand,model,transmission,fuelType
carID,,,,
69512,vw,golf,semi-auto,petrol
53000,toyota,yaris,manual,petrol
6366,audi,q2,semi-auto,petrol
29021,ford,fiesta,manual,petrol
10062,bmw,2 series,manual,petrol
...,...,...,...,...
37194,mercedes,c class,manual,petrol
6265,audi,q3,semi-auto,diesel
54886,toyota,aygo,automatic,petrol


In [ ]:
freq_brand = df_cat["Brand"].value_counts()
print(freq_brand)

Brand
ford        13966
mercedes    10088
vw           8987
opel         8130
bmw          6375
audi         6231
toyota       3931
skoda        3699
hyundai      2896
Name: count, dtype: int64


In [ ]:
model_freq = df_cat["model"].value_counts()
print(model_freq)

model
focus       6040
c class     4582
fiesta      3879
golf        2827
corsa       2008
            ... 
fox            1
200            1
s5             1
escort         1
terracan       1
Name: count, Length: 191, dtype: int64


In [ ]:
freq_trans = df_cat["transmission"].value_counts()
print(freq_trans)

transmission
manual       36418
semi-auto    14770
automatic    13115
Name: count, dtype: int64


In [ ]:
freq_fuel = df_cat["fuelType"].value_counts()
print(freq_fuel)

fuelType
petrol      35649
diesel      26714
hybrid       1937
electric        3
Name: count, dtype: int64


In [ ]:
freq_year = df["year"].value_counts()
print(freq_year)

year
2019    17560
2017    13970
2016     9941
2018     8900
2015     4918
2020     2708
2014     2515
2013     1689
2011      482
2012      404
2010      288
2009      203
2023      186
2008      130
2024      111
2007      105
2005       50
2006       49
2004       27
2003       25
2002       15
2001       12
2000        5
1999        5
1998        2
1996        1
1970        1
1997        1
Name: count, dtype: int64


In [ ]:
freq_damage = df["stated_no_damage"].value_counts()
print(freq_damage)

stated_no_damage
1    62979
0     1324
Name: count, dtype: int64


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 64303 entries, 69512 to 15795
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Brand             64303 non-null  object 
 1   model             64303 non-null  object 
 2   year              64303 non-null  int64  
 3   price             64303 non-null  int64  
 4   transmission      64303 non-null  object 
 5   mileage           64303 non-null  float64
 6   fuelType          64303 non-null  object 
 7   tax               64303 non-null  float64
 8   mpg               64303 non-null  float64
 9   engineSize        64303 non-null  float64
 10  paintQuality%     64303 non-null  float64
 11  previousOwners    64303 non-null  float64
 12  stated_no_damage  64303 non-null  int64  
dtypes: float64(6), int64(3), object(4)
memory usage: 6.9+ MB


In [ ]:
df.head(2)

NameError: name 'df' is not defined

In [ ]:
df = df.drop(["model"], axis=1)             #for the general model we don't care for the models, it's too many zero columns


In [ ]:
def Preprocessing(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    # Data Cleaning function
    # Takes the uncleaned dataframe and cleans all columns 
    # Avoids data leakage by training imputers only on training data.
    # Note: param fast should be left at True for the first submission, for the final submission we will experiment with more advanced imputation methods.
    X_train, X_test = clean_car_data(X_train, X_test, fast=True)
    
    # function call with one df
    num_cols = ["year", "mileage", "tax", "mpg", "engineSize",
                "paintQuality%", "previousOwners", "stated_no_damage"]
    cat_cols = ["Brand", "transmission", "fuelType"]
    
    scaler = StandardScaler()
    X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
    X_test[num_cols] = scaler.transform(X_test[num_cols])

    encoder = OneHotEncoder(sparse_output=False, dtype=int, handle_unknown='ignore')
    encoded_train = encoder.fit_transform(X_train[cat_cols])
    encoded_test = encoder.transform(X_test[cat_cols])
    
    encoded_train_df = pd.DataFrame(
        encoded_train, 
        columns=encoder.get_feature_names_out(cat_cols), 
        index=X_train.index)
    
    encoded_test_df = pd.DataFrame(
        encoded_test, 
        columns=encoder.get_feature_names_out(cat_cols), 
        index=X_test.index)
    

    X_train_encoded = pd.concat([X_train.drop(columns=cat_cols), encoded_train_df], axis=1)
    X_test_encoded = pd.concat([X_test.drop(columns=cat_cols), encoded_test_df], axis=1)

    X_test_encoded = X_test_encoded[X_train_encoded.columns]
    
    return X_train_encoded, X_test_encoded, scaler, encoder


In [3]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd

def cross_validation(X, y, model, n_folds=5, n_bins=10, metrics=None, random_state=42):
    """
    Cross-validation with stratification and preprocessing
    
    Parameters:
    -----------
    X : pd.DataFrame
        Features
    y : pd.Series or np.array
        Target variable
    model : object
        Model 
    n_folds : int
        Folds chosen
    n_bins : int
        bins for our stratification
    metrics : list of tuples
        metrics
        Es: [('MSE', mean_squared_error), ('MAE', mean_absolute_error)]
    random_state : int
        Seed 
        
    Returns:
    --------
    results : dict
        A dictionary:
        - 'fold_results': metrics for every fold
        - 'mean_metrics': mean of the metrics
        - 'std_metrics': standard deviation for the metrics
        - 'trained_models': models
    """
    if metrics is None:
        metrics = [
            ('MSE', mean_squared_error),
            ('RMSE', lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred))),
            ('MAE', mean_absolute_error),
            ('R2', r2_score)
        ]
    

    y_bins = pd.cut(y, bins=n_bins, labels=False)
    
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    
    fold_results = {metric_name: [] for metric_name, _ in metrics}
    fold_results_train = {metric_name: [] for metric_name, _ in metrics}  # Per train metrics
    trained_models = []

    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y_bins), 1):
        print(f"\n{'='*80}")
        print(f"FOLD {fold_idx}/{n_folds}")
        print(f"{'='*80}")
        
        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()
        y_train_fold = y.iloc[train_idx] if isinstance(y, pd.Series) else y[train_idx]
        y_val_fold = y.iloc[val_idx] if isinstance(y, pd.Series) else y[val_idx]
        
        print(f"Train size: {len(X_train_fold)}, Validation size: {len(X_val_fold)}")
        
        X_train_processed, X_val_processed, scaler, encoder = Preprocessing(
            X_train_fold, X_val_fold
        )
        

        model_fold = model.__class__(**model.get_params())
        model_fold.fit(X_train_processed, y_train_fold)
        

        y_pred_train = model_fold.predict(X_train_processed)
        y_pred_val = model_fold.predict(X_val_processed)
        
    
        print(f"\n📊 TRAIN Metrics:")
        print("-" * 40)
        for metric_name, metric_func in metrics:
            score_train = metric_func(y_train_fold, y_pred_train)
            fold_results_train[metric_name].append(score_train)
            print(f"  {metric_name:12s}: {score_train:12.4f}")
 
        print(f" VALIDATION Metrics:")
        print("-" * 40)
        for metric_name, metric_func in metrics:
            score_val = metric_func(y_val_fold, y_pred_val)
            fold_results[metric_name].append(score_val)
            print(f"  {metric_name:12s}: {score_val:12.4f}")
        

        trained_models.append({
            'model': model_fold,
            'scaler': scaler,
            'encoder': encoder
        })
    

    mean_metrics = {metric: np.mean(scores) for metric, scores in fold_results.items()}
    std_metrics = {metric: np.std(scores) for metric, scores in fold_results.items()}
    mean_metrics_train = {metric: np.mean(scores) for metric, scores in fold_results_train.items()}
    std_metrics_train = {metric: np.std(scores) for metric, scores in fold_results_train.items()}
    

    print(f"\n\n{'='*80}")
    print("Metrics for each fold")
    print(f"{'='*80}\n")
    

    print(f"{'Metric':<12s} | ", end="")
    for i in range(1, n_folds + 1):
        print(f"Fold {i:>2d}  ", end=" | ")
    print(f"{'Mean':>10s} | {'Std':>10s}")
    print("-" * 80)
    
    for metric_name in fold_results.keys():
        print(f"{metric_name:<12s} | ", end="")
        for score in fold_results[metric_name]:
            print(f"{score:>8.4f}", end=" | ")
        print(f"{mean_metrics[metric_name]:>10.4f} | {std_metrics[metric_name]:>10.4f}")
    
    print(f"\n{'='*80}")
    print("Results on validation set")
    print(f"{'='*80}")
    for metric_name in fold_results.keys():
        print(f"{metric_name:12s}: {mean_metrics[metric_name]:10.4f} (±{std_metrics[metric_name]:8.4f})")
    
    print(f"\n{'='*80}")
    print("Results on training set")
    print(f"{'='*80}")
    for metric_name in fold_results_train.keys():
        print(f"{metric_name:12s}: {mean_metrics_train[metric_name]:10.4f} (±{std_metrics_train[metric_name]:8.4f})")
    
    return {
        'fold_results_val': fold_results,
        'fold_results_train': fold_results_train,
        'mean_metrics_val': mean_metrics,
        'std_metrics_val': std_metrics,
        'mean_metrics_train': mean_metrics_train,
        'std_metrics_train': std_metrics_train,
        'trained_models': trained_models
    }



In [4]:
X = df.drop("price", axis=1)
y = df["price"]


n_bins = 10
y_bins = pd.cut(y, bins=n_bins, labels=False)

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=1,
    stratify=y_bins
)

NameError: name 'df' is not defined

In [ ]:
print(f"X_train shape is {X_train.shape}")
print(f"X_val size is {X_val.shape}")

X_train shape is (51442, 11)
X_val size is (12861, 11)


In [ ]:
pipe_linear = LinearRegression()

results = cross_validation(
    X=X_train,           
    y=y_train,
    model=pipe_linear,
    n_folds=10,
    n_bins=10,
    random_state=42
)

c:\Users\liber\anaconda3\envs\ML_env\Lib\site-packages\sklearn\model_selection\_split.py:811: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=10.
  warnings.warn(



FOLD 1/10
Train size: 46297, Validation size: 5145

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25015250.4122
  RMSE        :    5001.5248
  MAE         :    3076.8051
  R2          :       0.7377
 VALIDATION Metrics:
----------------------------------------
  MSE         : 25689684.3205
  RMSE        :    5068.4992
  MAE         :    3085.2523
  R2          :       0.7313

FOLD 2/10
Train size: 46297, Validation size: 5145

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25106217.3659
  RMSE        :    5010.6105
  MAE         :    3078.4775
  R2          :       0.7361
 VALIDATION Metrics:
----------------------------------------
  MSE         : 24871807.9159
  RMSE        :    4987.1643
  MAE         :    3075.3451
  R2          :       0.7456

FOLD 3/10
Train size: 46298, Validation size: 5144

📊 TRAIN Metrics:
----------------------------------------
  MSE         : 25167049.0616
  RMSE        :    5016.6771
  MAE         : 

In [ ]:
X_train_processed, X_val_processed, scaler, encoder = Preprocessing(X_train, X_val)

pipe_linear_final = LinearRegression()  
pipe_linear_final.fit(X_train_processed, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [ ]:
y_pred = pipe_linear_final.predict(X_val_processed)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print(f"\n{'='*50}")
print("FINAL VALIDATION RESULTS")
print(f"{'='*50}")
print(f"Validation MSE:  {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


print(f"\nCross-Validation MSE:  {results['mean_metrics_val']['MSE']:.3f}")
print(f"Cross-Validation RMSE: {results['mean_metrics_val']['RMSE']:.3f}")


FINAL VALIDATION RESULTS
Validation MSE:  23744931.593
Validation RMSE: 4872.877

Cross-Validation MSE:  25110972.472
Cross-Validation RMSE: 5010.210


Linear Regression

In [ ]:
pipe_linear_regression = Pipeline([("model", LinearRegression())])

In [ ]:
pipe_linear_regression.fit(X_train, y_train)
y_pred = pipe_linear_regression.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4822.507


Ridge

In [ ]:
pipe_ridge = Pipeline([("model", Ridge())])

In [ ]:
pipe_ridge.fit(X_train, y_train)
y_pred = pipe_ridge.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4822.423


Lasso

In [ ]:
pipe_lasso = Pipeline([("model", Lasso())])

In [ ]:
pipe_lasso.fit(X_train, y_train)
y_pred = pipe_lasso.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4823.400


ElasticNet

In [ ]:
pipe_elastic = Pipeline([("model", ElasticNet())])

In [ ]:
pipe_elastic.fit(X_train, y_train)
y_pred = pipe_elastic.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 8259.740


SVMRegressor

In [ ]:
pipe_SVM = Pipeline([("model", SVR())])

In [ ]:
pipe_SVM.fit(X_train, y_train)
y_pred = pipe_SVM.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

print(f"Validation MSE: {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


Validation MSE: 79452948.717
Validation RMSE: 8913.638


Random Forest

In [ ]:
pipe_rf = Pipeline([("model", RandomForestRegressor())])


In [ ]:
pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)

#print(f"Validation MSE: {mse:.3f}")
print(f"Validation RMSE: {rmse:.3f}")


Validation RMSE: 3218.916


Single tree

In [ ]:
pipe_tree = Pipeline([("model", DecisionTreeRegressor())])

In [ ]:
pipe_tree.fit(X_train, y_train)
y_pred = pipe_tree.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4390.532


KNRegressor

In [ ]:
pipe_KNR = pipe_tree = Pipeline([("model", KNeighborsRegressor())])

In [ ]:
pipe_KNR.fit(X_train, y_train)
y_pred = pipe_KNR.predict(X_val)

val_RMSE = root_mean_squared_error(y_val, y_pred)
print(f"Test RMSE: {val_RMSE:.3f}")

Test RMSE: 4589.610


Neural Network



In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.7, shuffle=True)

In [ ]:
model = nn.Sequential(
    nn.Linear
)

TypeError: torch.nn.modules.linear.Linear is not a Module subclass